In [ ]:
# Week 4 Day 4

# Week 4 Day 4

## FastAPI Model Serving Endpoint

**Name:** Chaitra MH

### Objective

To train a machine learning model, save the trained model, create a FastAPI prediction endpoint, and test the API using sample input data.

In [2]:
# Install FastAPI, Uvicorn, Joblib and LocalTunnel

!pip install fastapi uvicorn joblib -q

!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
changed 22 packages in 1s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼

In [3]:
# Import required libraries

import numpy as np
import joblib

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [5]:
# load the iris clssification datasets
iris =load_iris()
X=iris.data
y=iris.target
print("feature shape:", X.shape)
print("target shape:",y.shape)
print("\n Feature name:", iris.feature_names)
print("\n target classes:", iris.target_names)

feature shape: (150, 4)
target shape: (150,)

 Feature name: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

 target classes: ['setosa' 'versicolor' 'virginica']


In [9]:
# Split the dataset into training and testing data

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))

print("Testing samples:", len(X_test))

Training samples: 120
Testing samples: 30


In [11]:
# create and train the logistic regression model
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000, random_state=42)

In [12]:
# make prdections using the trained model
y_pred = model.predict(X_test)
#calculate model accuracy
accuracy =accuracy_score(
    y_test,
    y_pred
)
print("model accuracy:", accuracy)

model accuracy: 0.9666666666666667


In [13]:
#save the trained model using joblib
joblib.dump(
    model,
    "iris_model.pk1"
)
print("model saved successfully")

model saved successfully


In [14]:
import os
print(
    "model file exists:",
    os.path.exists("iris_model.pk1")
)

model file exists: True


In [15]:
%%writefile app.py

# Import FastAPI

from fastapi import FastAPI

# Import BaseModel for input validation

from pydantic import BaseModel

# Import Joblib to load the trained model

import joblib


# Create the FastAPI application

app = FastAPI(
    title="Iris Flower Prediction API",
    description="API for predicting Iris flower species",
    version="1.0"
)


# Load the trained machine learning model

model = joblib.load(
    "iris_model.pkl"
)


# Define the expected input structure

class IrisInput(BaseModel):

    sepal_length: float

    sepal_width: float

    petal_length: float

    petal_width: float


# Create the home endpoint

@app.get("/")

def home():

    return {
        "message": "Iris Prediction API is running"
    }


# Create the prediction endpoint

@app.post("/predict")

def predict(
    input_data: IrisInput
):

    # Convert API input into model format

    features = [[

        input_data.sepal_length,

        input_data.sepal_width,

        input_data.petal_length,

        input_data.petal_width

    ]]


    # Make prediction

    prediction = model.predict(
        features
    )[0]


    # Get prediction probabilities

    probabilities = model.predict_proba(
        features
    )[0]


    # Iris class names

    class_names = [

        "setosa",

        "versicolor",

        "virginica"

    ]


    # Return prediction result

    return {

        "predicted_class": int(
            prediction
        ),

        "predicted_species": class_names[
            prediction
        ],

        "probabilities": probabilities.tolist()

    }

Writing app.py


In [16]:
!cat app.py


# Import FastAPI

from fastapi import FastAPI

# Import BaseModel for input validation

from pydantic import BaseModel

# Import Joblib to load the trained model

import joblib


# Create the FastAPI application

app = FastAPI(
    title="Iris Flower Prediction API",
    description="API for predicting Iris flower species",
    version="1.0"
)


# Load the trained machine learning model

model = joblib.load(
    "iris_model.pkl"
)


# Define the expected input structure

class IrisInput(BaseModel):

    sepal_length: float

    sepal_width: float

    petal_length: float

    petal_width: float


# Create the home endpoint

@app.get("/")

def home():

    return {
        "message": "Iris Prediction API is running"
    }


# Create the prediction endpoint

@app.post("/predict")

def predict(
    input_data: IrisInput
):

    # Convert API input into model format

    features = [[

        input_data.sepal_length,

        input_data.sepal_width,

        input_data.petal_length,

   

In [21]:
import os

os.rename(
    "/content/iris_model.pk1",
    "/content/iris_model.pkl"
)

print("File renamed successfully.")

File renamed successfully.


In [22]:
print(os.listdir("/content"))

['.config', 'iris_model.pkl', 'app.py', '__pycache__', 'sample_data']


In [23]:
model = joblib.load("/content/iris_model.pkl")

In [24]:
!uvicorn app:app --host 0.0.0.0 --port 8000 &

INFO:     Started server process [8473]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [8473]


In [27]:
!pkill -f uvicorn

In [29]:
!nohup uvicorn app:app --host 0.0.0.0 --port 8000 > uvicorn.log 2>&1 &

In [30]:
!cat uvicorn.log

INFO:     Started server process [10864]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [31]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/"
)

print("Status code:", response.status_code)

print("Response:", response.json())

Status code: 200
Response: {'message': 'Iris Prediction API is running'}


In [32]:
import requests

# Sample Iris flower measurements

sample_data = {
    "sepal_length": 5.1,
    "sepal_width": 3.5,
    "petal_length": 1.4,
    "petal_width": 0.2
}

# Send the input data to the FastAPI prediction endpoint

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=sample_data
)

# Display the API response

print("Status Code:", response.status_code)
print("Prediction Result:", response.json())

Status Code: 200
Prediction Result: {'predicted_class': 0, 'predicted_species': 'setosa', 'probabilities': [0.9783999136173778, 0.021600031295543935, 5.508707825600511e-08]}


In [33]:
import requests

# Second sample Iris flower measurements

sample_data_2 = {
    "sepal_length": 6.7,
    "sepal_width": 3.0,
    "petal_length": 5.2,
    "petal_width": 2.3
}

# Send the data to the API

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=sample_data_2
)

# Display the result

print("Status Code:", response.status_code)
print("Prediction Result:", response.json())

Status Code: 200
Prediction Result: {'predicted_class': 2, 'predicted_species': 'virginica', 'probabilities': [8.387514874689573e-05, 0.09254469468605948, 0.9073714301651936]}


In [34]:
import requests

def test_prediction_endpoint():

    # Test input

    test_input = {
        "sepal_length": 5.1,
        "sepal_width": 3.5,
        "petal_length": 1.4,
        "petal_width": 0.2
    }

    # Send request to the prediction endpoint

    response = requests.post(
        "http://127.0.0.1:8000/predict",
        json=test_input
    )

    # Test the status code

    assert response.status_code == 200

    # Get the response

    result = response.json()

    # Check required response fields

    assert "predicted_class" in result
    assert "predicted_species" in result
    assert "probabilities" in result

    # Display successful test message

    print("API test passed successfully!")


# Run the test

test_prediction_endpoint()

API test passed successfully!


In [35]:
invalid_data = {
    "sepal_length": 5.1,
    "sepal_width": 3.5
}

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=invalid_data
)

print("Status Code:", response.status_code)
print("Response:", response.json())

Status Code: 422
Response: {'detail': [{'type': 'missing', 'loc': ['body', 'petal_length'], 'msg': 'Field required', 'input': {'sepal_length': 5.1, 'sepal_width': 3.5}}, {'type': 'missing', 'loc': ['body', 'petal_width'], 'msg': 'Field required', 'input': {'sepal_length': 5.1, 'sepal_width': 3.5}}]}


In [ ]:
## Conclusion

In this task, a Logistic Regression model was trained using the Iris dataset and saved using Joblib.

A FastAPI application was created to serve the trained machine learning model through a REST API. The `/predict` endpoint accepted four Iris flower measurements and returned the predicted flower species and prediction probabilities.

The FastAPI home endpoint and prediction endpoint were tested successfully. Input validation was handled using Pydantic, and the API returned valid predictions with HTTP status code 200.
